# Step 2: QAOA on Bloqade — 4 Assets, 4 Qubits
## YQuantum 2026 — Capgemini / The Hartford Challenge

**What this notebook does:**
1. Takes the QUBO/Ising from Step 1
2. Builds QAOA circuit in Bloqade (squin)
3. Optimizes gamma/beta parameters
4. Shows the quantum probability distribution
5. Compares quantum results vs brute-force ground truth
6. Adds noise analysis

**Ground truth from Step 1:** x = [1, 0, 0, 1] (Equities US + Gov Bonds), energy = -0.006004

---

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from bloqade import squin
from bloqade.pyqrack import StackMemorySimulator

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('Bloqade imported successfully!')

ModuleNotFoundError: No module named 'bloqade'

## 1. Ising Parameters from Step 1

From our QUBO (q=1.0, λ=10.0, B=0.30):

$$H_{\text{Ising}} = \sum_{i<j} J_{ij} \, s_i s_j + \sum_i h_i \, s_i$$

QAOA encodes this as:
- **Cost layer:** ZZ rotations for each J coupling + Z rotations for each h field
- **Mixer layer:** Rx rotations on all qubits

In [ ]:
# Ising parameters from Step 1
n_qubits = 4
asset_ids = ['A004', 'A047', 'A026', 'A017']
sectors = ['Equities US', 'HY Credit', 'IG Credit', 'Gov Bonds']

# QUBO matrix
Q = np.array([[-0.422395,  0.048003,  0.096007,  0.160002],
              [ 0.048003, -0.328333,  0.072004,  0.120003],
              [ 0.096007,  0.072004, -0.57997 ,  0.240006],
              [ 0.160002,  0.120003,  0.240006, -0.803612]])
qubo_offset = 0.9

# J couplings (6 edges for fully connected 4-qubit graph)
J_couplings = [
    (0, 1, 0.012001),
    (0, 2, 0.024002),
    (0, 3, 0.040000),
    (1, 2, 0.018001),
    (1, 3, 0.030001),
    (2, 3, 0.060001),
]

# Local fields
h_fields = [-0.135194, -0.104164, -0.187981, -0.271804]

# Max weights for decoding
w_max = np.array([0.08, 0.06, 0.12, 0.20])
mu = np.array([0.0823, 0.0730, 0.0334, 0.0183])

# Ground truth
ground_truth = [1, 0, 0, 1]
ground_energy = -0.006004

print(f'4-qubit Ising problem loaded')
print(f'  {len(J_couplings)} J couplings (fully connected)')
print(f'  {n_qubits} local fields h')
print(f'  Ground truth: {ground_truth} (energy: {ground_energy})')

## 2. QAOA Circuit in Bloqade

QAOA has `p` layers, each with:
1. **Cost layer** (parameter γ): applies `exp(-i*γ*H_cost)`
   - ZZ interaction for each J coupling: decomposed as H → CZ → Rz(γ*J) → CZ → H  
   - Z rotation for each h field: Rz(γ*h)
2. **Mixer layer** (parameter β): applies `exp(-i*β*H_mixer)`
   - Rx(2β) on every qubit

In [ ]:
def make_qaoa_circuit(gamma_list, beta_list, J_couplings, h_fields):
    """Build a p-layer QAOA circuit for the portfolio Ising Hamiltonian."""
    p = len(gamma_list)
    n = len(h_fields)
    
    @squin.kernel
    def qaoa_circuit():
        q = squin.qalloc(n)
        
        # Initial superposition: |+>^n
        for i in range(n):
            squin.h(q[i])
        
        # QAOA layers
        for layer in range(p):
            gamma = gamma_list[layer]
            beta = beta_list[layer]
            
            # === Cost layer ===
            # ZZ interactions: exp(-i * gamma * J_ij * Z_i Z_j)
            # Decomposition: CNOT → Rz(2*gamma*J) → CNOT
            for (i, j, j_val) in J_couplings:
                squin.cx(q[i], q[j])
                squin.rz(2 * gamma * j_val, q[j])
                squin.cx(q[i], q[j])
            
            # Z rotations: exp(-i * gamma * h_i * Z_i)
            for i in range(n):
                squin.rz(2 * gamma * h_fields[i], q[i])
            
            # === Mixer layer ===
            # Rx rotations: exp(-i * beta * X_i)
            for i in range(n):
                squin.rx(2 * beta, q[i])
        
        # Measure all qubits
        bits = squin.broadcast.measure(q)
        return bits
    
    return qaoa_circuit


# Test with random parameters
test_circuit = make_qaoa_circuit([0.5], [0.5], J_couplings, h_fields)
print('QAOA circuit builder works!')
print(f'  1 layer: 6 ZZ gates + 4 Rz gates + 4 Rx gates = 14 parameterized gates')

## 3. Run QAOA and Measure

In [ ]:
def run_qaoa(gamma_list, beta_list, J_couplings, h_fields, shots=1000):
    """Run QAOA circuit and return measurement counts."""
    n = len(h_fields)
    circuit = make_qaoa_circuit(gamma_list, beta_list, J_couplings, h_fields)
    
    simulator = StackMemorySimulator(min_qubits=n)
    task = simulator.task(circuit)
    
    # Collect measurements
    counts = {}
    for _ in range(shots):
        result = task.run()
        # Convert measurement result to bitstring
        bitstring = ''.join(str(int(b)) for b in result)
        counts[bitstring] = counts.get(bitstring, 0) + 1
    
    return counts


def evaluate_qubo(bitstring, Q, offset):
    """Evaluate QUBO energy for a bitstring."""
    x = np.array([int(b) for b in bitstring], dtype=float)
    return float(x @ Q @ x + offset)


def qaoa_expected_energy(gamma_list, beta_list, J_couplings, h_fields, Q, offset, shots=500):
    """Compute expected energy from QAOA measurements."""
    counts = run_qaoa(gamma_list, beta_list, J_couplings, h_fields, shots)
    total_energy = 0
    total_shots = sum(counts.values())
    for bitstring, count in counts.items():
        energy = evaluate_qubo(bitstring, Q, offset)
        total_energy += energy * count
    return total_energy / total_shots


# Quick test
test_counts = run_qaoa([0.5], [0.5], J_couplings, h_fields, shots=100)
print(f'Test run: {len(test_counts)} unique states observed')
for bs, count in sorted(test_counts.items(), key=lambda x: -x[1])[:5]:
    energy = evaluate_qubo(bs, Q, qubo_offset)
    print(f'  {bs}: {count} counts, energy={energy:.4f}')

## 4. Optimize QAOA Parameters

Use classical optimization (scipy) to find the best γ, β that minimize expected energy.

In [ ]:
def optimize_qaoa(p, J_couplings, h_fields, Q, offset, shots=300, maxiter=50):
    """Optimize QAOA parameters for p layers."""
    
    def cost_function(params):
        gamma_list = params[:p].tolist()
        beta_list = params[p:].tolist()
        energy = qaoa_expected_energy(gamma_list, beta_list, J_couplings, h_fields, Q, offset, shots)
        return energy
    
    # Initial parameters
    x0 = np.random.uniform(0, np.pi, 2 * p)
    
    print(f'Optimizing QAOA with p={p} layers ({2*p} parameters)...')
    
    # Track progress
    history = []
    def callback(xk):
        e = cost_function(xk)
        history.append(e)
        if len(history) % 5 == 0:
            print(f'  Iteration {len(history)}: energy = {e:.6f}')
    
    result = minimize(cost_function, x0, method='COBYLA',
                     options={'maxiter': maxiter, 'rhobeg': 0.5},
                     callback=callback)
    
    opt_gamma = result.x[:p].tolist()
    opt_beta = result.x[p:].tolist()
    
    print(f'\nOptimization complete!')
    print(f'  Best energy: {result.fun:.6f} (ground truth: {ground_energy})')
    print(f'  Optimal gamma: {[round(g, 4) for g in opt_gamma]}')
    print(f'  Optimal beta:  {[round(b, 4) for b in opt_beta]}')
    
    return opt_gamma, opt_beta, history


# Optimize with p=1 layer first
opt_gamma_1, opt_beta_1, hist_1 = optimize_qaoa(1, J_couplings, h_fields, Q, qubo_offset)

In [ ]:
# Optimize with p=2 layers
opt_gamma_2, opt_beta_2, hist_2 = optimize_qaoa(2, J_couplings, h_fields, Q, qubo_offset)

In [ ]:
# Plot optimization convergence
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hist_1, 'o-', label=f'p=1 (final: {hist_1[-1]:.4f})', markersize=4)
ax.plot(hist_2, 's-', label=f'p=2 (final: {hist_2[-1]:.4f})', markersize=4)
ax.axhline(ground_energy, color='red', linestyle='--', linewidth=2, label=f'Ground truth: {ground_energy}')
ax.set_xlabel('Iteration')
ax.set_ylabel('Expected Energy')
ax.set_title('QAOA Optimization Convergence')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Probability Distribution (The Key Result)

Run the optimized QAOA circuit with many shots and visualize the probability of each portfolio state.

In [ ]:
def plot_qaoa_distribution(gamma_list, beta_list, J_couplings, h_fields, 
                            Q, offset, shots, title, ax):
    """Run QAOA and plot the probability distribution."""
    counts = run_qaoa(gamma_list, beta_list, J_couplings, h_fields, shots)
    
    # All 16 states
    all_states = [format(i, f'0{n_qubits}b') for i in range(2**n_qubits)]
    all_counts = [counts.get(s, 0) for s in all_states]
    probs = np.array(all_counts) / shots
    energies = [evaluate_qubo(s, Q, offset) for s in all_states]
    
    # Color: green=ground truth, red=invest nothing, blue gradient by energy
    gt_str = ''.join(map(str, ground_truth))
    colors = []
    for s in all_states:
        if s == gt_str:
            colors.append('#27ae60')
        elif s == '0000':
            colors.append('#e74c3c')
        elif evaluate_qubo(s, Q, offset) < 0:
            colors.append('#2980b9')
        else:
            colors.append('#bdc3c7')
    
    ax.bar(range(16), probs * 100, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_xticks(range(16))
    ax.set_xticklabels(all_states, rotation=45, fontsize=8)
    ax.set_ylabel('Probability (%)')
    ax.set_title(title)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Find most probable state
    best_idx = np.argmax(all_counts)
    best_state = all_states[best_idx]
    best_energy = energies[best_idx]
    
    return counts, best_state, best_energy


# Run with many shots for both p=1 and p=2
shots = 2000

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

counts_p1, best_p1, energy_p1 = plot_qaoa_distribution(
    opt_gamma_1, opt_beta_1, J_couplings, h_fields, Q, qubo_offset,
    shots, f'QAOA p=1 ({shots} shots)', axes[0])

counts_p2, best_p2, energy_p2 = plot_qaoa_distribution(
    opt_gamma_2, opt_beta_2, J_couplings, h_fields, Q, qubo_offset,
    shots, f'QAOA p=2 ({shots} shots)', axes[1])

plt.tight_layout()
plt.show()

gt_str = ''.join(map(str, ground_truth))
print(f'Ground truth state: {gt_str} (energy: {ground_energy})')
print(f'\np=1: Most frequent = {best_p1} (energy: {energy_p1:.4f}), '
      f'ground truth probability = {counts_p1.get(gt_str, 0)/shots*100:.1f}%')
print(f'p=2: Most frequent = {best_p2} (energy: {energy_p2:.4f}), '
      f'ground truth probability = {counts_p2.get(gt_str, 0)/shots*100:.1f}%')

## 6. Gamma-Beta Landscape (p=1)

Scan γ and β to visualize the energy landscape and confirm our optimizer found a good minimum.

In [ ]:
# Scan gamma-beta landscape
n_points = 20
gamma_range = np.linspace(0, 2*np.pi, n_points)
beta_range = np.linspace(0, np.pi, n_points)

energy_landscape = np.zeros((n_points, n_points))

print(f'Scanning {n_points}x{n_points} = {n_points**2} parameter combinations...')
for gi, gamma in enumerate(gamma_range):
    for bi, beta in enumerate(beta_range):
        energy_landscape[bi, gi] = qaoa_expected_energy(
            [gamma], [beta], J_couplings, h_fields, Q, qubo_offset, shots=100)
    if (gi+1) % 5 == 0:
        print(f'  {gi+1}/{n_points} columns done')

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(energy_landscape, extent=[0, 2*np.pi, np.pi, 0],
               aspect='auto', cmap='viridis', origin='upper')
plt.colorbar(im, ax=ax, label='Expected Energy')

# Mark optimized point
ax.plot(opt_gamma_1[0] % (2*np.pi), opt_beta_1[0] % np.pi, 
        'r*', markersize=20, label=f'Optimized ({opt_gamma_1[0]:.2f}, {opt_beta_1[0]:.2f})')

ax.set_xlabel('γ (gamma)')
ax.set_ylabel('β (beta)')
ax.set_title('QAOA Energy Landscape (p=1)\nDarker = Lower Energy = Better')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

## 7. Decode the Quantum Result → Portfolio

In [ ]:
# Use the best QAOA result (p=2)
final_counts = run_qaoa(opt_gamma_2, opt_beta_2, J_couplings, h_fields, shots=5000)

# Sort by frequency
sorted_counts = sorted(final_counts.items(), key=lambda x: -x[1])

print(f'=== QAOA Portfolio Results (5000 shots, p=2) ===')
print(f'{"Rank":>4} {"State":>6} {"Counts":>7} {"Prob%":>7} {"Energy":>9} '
      f'{"Assets":>25} {"Σw":>6} {"Return":>9}')
print('-' * 85)

for rank, (bitstring, count) in enumerate(sorted_counts[:10]):
    x = np.array([int(b) for b in bitstring], dtype=float)
    w = x * w_max
    energy = evaluate_qubo(bitstring, Q, qubo_offset)
    ret = float(mu @ w)
    selected = [f'{asset_ids[i]}' for i in range(n_qubits) if x[i] == 1]
    marker = ' ◀ GROUND TRUTH' if bitstring == gt_str else ''
    
    print(f"{rank+1:>4} {bitstring:>6} {count:>7} {count/5000*100:>6.1f}% {energy:>9.4f} "
          f"{', '.join(selected):>25} {w.sum():>6.3f} {ret:>9.6f}{marker}")

## 8. Quantum vs Classical Comparison

In [ ]:
# Compare quantum probability vs brute-force energy ranking
all_states = [format(i, f'0{n_qubits}b') for i in range(2**n_qubits)]
brute_energies = [(s, evaluate_qubo(s, Q, qubo_offset)) for s in all_states]
brute_energies.sort(key=lambda x: x[1])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Brute-force energy ranking
bf_states = [s for s, e in brute_energies]
bf_energies = [e for s, e in brute_energies]
bf_colors = ['#27ae60' if s == gt_str else '#e74c3c' if s == '0000' 
             else '#2980b9' if e < 0 else '#bdc3c7' for s, e in brute_energies]

axes[0].bar(range(16), bf_energies, color=bf_colors, edgecolor='black', linewidth=0.5)
axes[0].set_xticks(range(16))
axes[0].set_xticklabels(bf_states, rotation=45, fontsize=8)
axes[0].set_ylabel('QUBO Energy')
axes[0].set_title('Classical: Energy Ranking (brute force)')
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].grid(True, alpha=0.3, axis='y')

# Right: QAOA probability (same state ordering)
qaoa_probs = [final_counts.get(s, 0) / 5000 * 100 for s in bf_states]

axes[1].bar(range(16), qaoa_probs, color=bf_colors, edgecolor='black', linewidth=0.5)
axes[1].set_xticks(range(16))
axes[1].set_xticklabels(bf_states, rotation=45, fontsize=8)
axes[1].set_ylabel('QAOA Probability (%)')
axes[1].set_title('Quantum: QAOA Measurement Probability (p=2)')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('Green = ground truth, Red = invest nothing, Blue = negative energy states')
print('\nIf QAOA works: the green bar should be the tallest on the right plot,')
print('matching the lowest bar on the left plot.')

## 9. Noise Analysis

Compare noiseless QAOA vs noisy simulation to see how noise degrades performance.

In [ ]:
def make_noisy_qaoa(gamma_list, beta_list, J_couplings, h_fields, noise_p):
    """Build QAOA circuit with depolarizing noise after each gate."""
    p_layers = len(gamma_list)
    n = len(h_fields)
    
    @squin.kernel
    def noisy_qaoa():
        q = squin.qalloc(n)
        
        for i in range(n):
            squin.h(q[i])
            squin.depolarize(p=noise_p, qubit=q[i])
        
        for layer in range(p_layers):
            gamma = gamma_list[layer]
            beta = beta_list[layer]
            
            for (i, j, j_val) in J_couplings:
                squin.cx(q[i], q[j])
                squin.depolarize2(p=noise_p, control=q[i], target=q[j])
                squin.rz(2 * gamma * j_val, q[j])
                squin.depolarize(p=noise_p, qubit=q[j])
                squin.cx(q[i], q[j])
                squin.depolarize2(p=noise_p, control=q[i], target=q[j])
            
            for i in range(n):
                squin.rz(2 * gamma * h_fields[i], q[i])
                squin.depolarize(p=noise_p, qubit=q[i])
            
            for i in range(n):
                squin.rx(2 * beta, q[i])
                squin.depolarize(p=noise_p, qubit=q[i])
        
        bits = squin.broadcast.measure(q)
        return bits
    
    return noisy_qaoa


def run_noisy_qaoa(gamma_list, beta_list, J_couplings, h_fields, noise_p, shots=2000):
    """Run noisy QAOA and return counts."""
    circuit = make_noisy_qaoa(gamma_list, beta_list, J_couplings, h_fields, noise_p)
    simulator = StackMemorySimulator(min_qubits=len(h_fields))
    task = simulator.task(circuit)
    
    counts = {}
    for _ in range(shots):
        result = task.run()
        bitstring = ''.join(str(int(b)) for b in result)
        counts[bitstring] = counts.get(bitstring, 0) + 1
    return counts


# Compare noise levels
noise_levels = [0.0, 0.005, 0.01, 0.02, 0.05, 0.1]
shots = 2000

gt_probs = []
avg_energies = []

print('=== Noise Scan ===')
print(f'{"Noise p":>8} {"GT Prob%":>9} {"Avg Energy":>11} {"Best State":>11}')
print('-' * 45)

for noise_p in noise_levels:
    if noise_p == 0:
        counts = run_qaoa(opt_gamma_2, opt_beta_2, J_couplings, h_fields, shots)
    else:
        counts = run_noisy_qaoa(opt_gamma_2, opt_beta_2, J_couplings, h_fields, noise_p, shots)
    
    gt_prob = counts.get(gt_str, 0) / shots * 100
    gt_probs.append(gt_prob)
    
    avg_e = sum(evaluate_qubo(bs, Q, qubo_offset) * c for bs, c in counts.items()) / shots
    avg_energies.append(avg_e)
    
    best_state = max(counts, key=counts.get)
    print(f'{noise_p:>8.3f} {gt_prob:>8.1f}% {avg_e:>11.6f} {best_state:>11}')

print(f'\nAs noise increases, ground truth probability drops and energy increases.')

In [ ]:
# Plot noise impact
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(noise_levels, gt_probs, 'o-', color='#27ae60', linewidth=2, markersize=8)
axes[0].set_xlabel('Depolarizing Noise Probability')
axes[0].set_ylabel('Ground Truth Probability (%)')
axes[0].set_title('Ground Truth Probability vs Noise')
axes[0].grid(True, alpha=0.3)

axes[1].plot(noise_levels, avg_energies, 's-', color='#e74c3c', linewidth=2, markersize=8)
axes[1].axhline(ground_energy, color='green', linestyle='--', label=f'Ground truth: {ground_energy}')
axes[1].set_xlabel('Depolarizing Noise Probability')
axes[1].set_ylabel('Average Energy')
axes[1].set_title('Expected Energy vs Noise')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Summary

In [ ]:
print('=== Step 2 Complete ===')
print(f'\nProblem: 4-asset portfolio optimization')
print(f'Assets: {asset_ids} ({sectors})')
print(f'Qubits: {n_qubits}')
print(f'\nQAOA Results:')
print(f'  p=1: Best state = {best_p1}, ground truth prob = {counts_p1.get(gt_str,0)/2000*100:.1f}%')
print(f'  p=2: Best state = {best_p2}, ground truth prob = {counts_p2.get(gt_str,0)/2000*100:.1f}%')
print(f'  Ground truth: {gt_str} (energy: {ground_energy})')
print(f'\nNoise Analysis:')
print(f'  Noiseless GT prob: {gt_probs[0]:.1f}%')
print(f'  Noise=0.01 GT prob: {gt_probs[2]:.1f}%')
print(f'  Noise=0.05 GT prob: {gt_probs[4]:.1f}%')
print(f'\nNext: Scale to 8 qubits (8 assets or 4 assets x 2 qubits)')